# ASL Recognition — Optical Flow Feature Extraction (Steps 4–7)

**Member 2 deliverable.** This notebook picks up where `ASL_Colab_Setup.ipynb` (Member 1) leaves off.

**Pipeline implemented here:**
1. Re-download preprocessed frames from Box (same source Member 1 used)
2. Load `nslt_100.json` to map each video → class index + train/val/test split
3. **Step 4** — Compute Farnebäck dense optical flow between consecutive frames
4. **Step 5** — Visualize sample flows (HSV-encoded) for the report
5. **Step 6** — Extract per-video features (HOOF + magnitude histogram + stats)
6. **Step 7** — Save `features.npy`, `labels.npy`, `splits.json` for Member 3

**Hand-off to Member 3:** the artifacts in `artifacts/` are everything they need to train the classifier (Step 8 onwards) on their laptop without re-running this pipeline.

## 0. Setup — Install dependencies + fetch metadata + download frames

In [ ]:
!pip install -q opencv-python numpy matplotlib tqdm requests

In [ ]:
# Pull metadata file from the repo (it lives under data/ in the repo)
import os, urllib.request

url = 'https://raw.githubusercontent.com/TanviDeore/ASL_Recognition_Grp14/main/data/nslt_100.json'
urllib.request.urlretrieve(url, 'nslt_100.json')

assert os.path.exists('nslt_100.json') and os.path.getsize('nslt_100.json') > 0, 'Download failed'
print(f'Downloaded nslt_100.json — {os.path.getsize("nslt_100.json")/1024:.1f} KB')

In [ ]:
# Download preprocessed frames from Box — identical URL to Member 1's notebook.
# Skips the download if frames are already present (re-run safe).
import os, requests, zipfile
from tqdm import tqdm

def download_data():
    url = 'https://utdallas.box.com/shared/static/5ry2jpk80fxvdq9xet51q65o3sd6bx0v.zip'
    output_zip = '/content/asl_frames.zip'
    extract_to = 'data/frames'

    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'Frames already present at {extract_to} ({len(os.listdir(extract_to))} folders) — skipping download.')
        return

    os.makedirs('data', exist_ok=True)
    print('Downloading dataset from Box...')
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    with open(output_zip, 'wb') as f, tqdm(
        total=total_size, unit='iB', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            bar.update(f.write(chunk))

    print('\nExtracting frames...')
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    os.remove(output_zip)
    print(f'Done. {len(os.listdir(extract_to))} folders ready in {extract_to}')

download_data()

## 1. Build the work list

`nslt_100.json` has 2038 video entries, but only ~1013 folders were extracted (the rest are listed in `missing.txt` from the WLASL source). We process only the videos we actually have frames for.

In [ ]:
import json
from collections import Counter

with open('nslt_100.json') as f:
    meta = json.load(f)

frames_root = 'data/frames'
available_video_ids = set(os.listdir(frames_root))

work_list = []
for vid, info in meta.items():
    if vid not in available_video_ids:
        continue
    work_list.append({
        'video_id': vid,
        'class_idx': info['action'][0],
        'subset': info.get('subset', 'unknown'),
    })

print(f'Entries in nslt_100.json: {len(meta)}')
print(f'Folders available in data/frames: {len(available_video_ids)}')
print(f'Videos to process: {len(work_list)}')
print(f'Subset distribution: {dict(Counter(w["subset"] for w in work_list))}')
print(f'Number of unique classes: {len(set(w["class_idx"] for w in work_list))}')

## 2. Step 4 — Farnebäck Dense Optical Flow

**Why dense optical flow?** It estimates a motion vector `(dx, dy)` for *every pixel* between two consecutive frames. For ASL, this captures the entire hand-and-arm motion field — not just sparse keypoints — which is what we want to classify gestures with.

**Why Farnebäck specifically?** It's the OpenCV built-in, runs on CPU at ~30 FPS, and is the canonical choice for classical optical-flow pipelines. RAFT (deep learning) would be more accurate but needs a GPU and adds dependencies — overkill for a feature pipeline that already has thousands of frame pairs.

**Why we resize to 160×120 first:** raw frames are larger; Farnebäck cost scales with pixel count. 160×120 keeps gesture-scale motion intact and runs ~6x faster than full-res. This is the standard cheat for HOOF-based recognition.

In [ ]:
import cv2
import numpy as np

FLOW_SIZE = (160, 120)  # (width, height) — what we resize each frame to before computing flow

def compute_flow(prev_gray, next_gray):
    """Farnebäck dense optical flow. Returns (H, W, 2) array of (dx, dy) per pixel."""
    return cv2.calcOpticalFlowFarneback(
        prev_gray, next_gray, None,
        pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
    )

def load_video_frames(video_dir):
    """Load all .jpg frames in a folder, sorted by filename."""
    files = sorted(f for f in os.listdir(video_dir) if f.endswith('.jpg'))
    frames = []
    for fn in files:
        img = cv2.imread(os.path.join(video_dir, fn))
        if img is not None:
            frames.append(img)
    return frames

def video_flows(video_dir):
    """For a video folder, return list of flow arrays — one per consecutive frame pair."""
    frames = load_video_frames(video_dir)
    if len(frames) < 2:
        return []
    flows = []
    prev_gray = cv2.cvtColor(cv2.resize(frames[0], FLOW_SIZE), cv2.COLOR_BGR2GRAY)
    for i in range(1, len(frames)):
        next_gray = cv2.cvtColor(cv2.resize(frames[i], FLOW_SIZE), cv2.COLOR_BGR2GRAY)
        flows.append(compute_flow(prev_gray, next_gray))
        prev_gray = next_gray
    return flows

# Quick sanity check on one video
sample_dir = os.path.join(frames_root, work_list[0]['video_id'])
flows = video_flows(sample_dir)
print(f'Video {work_list[0]["video_id"]}: {len(flows)} flow pairs, each of shape {flows[0].shape}')

## 3. Step 5 — Visualize Optical Flow (figures for the report)

We encode flow as an HSV image and convert to BGR:
- **Hue** = direction of motion (angle)
- **Saturation** = constant 255
- **Value** = magnitude of motion (normalized)

Result: stationary regions are black, moving regions glow with a color that indicates direction. This is **Figure 2** in the report (per the implementation plan).

In [ ]:
import matplotlib.pyplot as plt

def flow_to_rgb(flow):
    h, w = flow.shape[:2]
    hsv = np.zeros((h, w, 3), dtype=np.uint8)
    hsv[..., 1] = 255
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = (ang * 180 / np.pi / 2).astype(np.uint8)
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

os.makedirs('visualizations', exist_ok=True)

# Pick one video each from 4 different classes
samples, seen = [], set()
for w in work_list:
    if w['class_idx'] not in seen:
        samples.append(w)
        seen.add(w['class_idx'])
    if len(samples) == 4:
        break

for sv in samples:
    video_dir = os.path.join(frames_root, sv['video_id'])
    frames = load_video_frames(video_dir)
    if len(frames) < 2:
        continue
    mid = len(frames) // 2
    f1 = cv2.resize(frames[mid - 1], FLOW_SIZE)
    f2 = cv2.resize(frames[mid], FLOW_SIZE)
    flow = compute_flow(
        cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY),
        cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY),
    )
    flow_rgb = flow_to_rgb(flow)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    axes[0].imshow(cv2.cvtColor(f1, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Frame t  (vid {sv["video_id"]}, class {sv["class_idx"]})')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(f2, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Frame t+1')
    axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(flow_rgb, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Optical flow (HSV-mapped)')
    axes[2].axis('off')
    save_path = f'visualizations/flow_class{sv["class_idx"]:02d}_vid{sv["video_id"]}.png'
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=130)
    plt.show()
    print(f'Saved {save_path}')

## 4. Step 6 — Feature Extraction (HOOF + Magnitude Histogram + Stats)

We turn each flow field (160 × 120 × 2 = 38,400 numbers) into a fixed-size feature vector.

**HOOF (Histogram of Oriented Optical Flow)** — 8 angle bins, each pixel contributes its magnitude as the weight. Captures *which directions* the motion is pointing in this frame pair. Standard for optical-flow gesture recognition.

**Magnitude histogram** — 8 bins. Captures *how much* motion there is (slow vs fast gestures).

**Stats** — mean, std, max of magnitude + mean angle. Cheap global summary.

Total = **20 dims per frame pair**. We then **average across all pairs** in a video to get a single 20-dim vector per video. Averaging is robust to videos of slightly different lengths and is the cleanest input for an SVM.

In [ ]:
ANGLE_BINS = 8
MAG_BINS = 8
FEATURE_DIM = ANGLE_BINS + MAG_BINS + 4  # 20

def per_pair_features(flow):
    """Convert one flow field (H, W, 2) into a 20-dim feature vector."""
    fx, fy = flow[..., 0], flow[..., 1]
    mag, ang = cv2.cartToPolar(fx, fy)  # ang in radians [0, 2π)
    mag_flat = mag.ravel()
    ang_flat = ang.ravel()

    # HOOF — angle histogram weighted by magnitude, then L1-normalized
    hoof, _ = np.histogram(ang_flat, bins=ANGLE_BINS, range=(0, 2 * np.pi), weights=mag_flat)
    if hoof.sum() > 0:
        hoof = hoof / hoof.sum()

    # Magnitude histogram (clip top 1% to suppress outliers from flow errors)
    mag_clipped = np.clip(mag_flat, 0, np.percentile(mag_flat, 99) + 1e-6)
    mhist, _ = np.histogram(mag_clipped, bins=MAG_BINS)
    if mhist.sum() > 0:
        mhist = mhist / mhist.sum()

    stats = np.array([mag.mean(), mag.std(), mag.max(), ang.mean()])
    return np.concatenate([hoof, mhist, stats])

def video_features(video_dir):
    flows = video_flows(video_dir)
    if not flows:
        return None
    per_pair = np.stack([per_pair_features(f) for f in flows])  # (N_pairs, 20)
    return per_pair.mean(axis=0).astype(np.float32)

# Sanity check on one video
feat = video_features(os.path.join(frames_root, work_list[0]['video_id']))
print(f'Feature dim: {len(feat)} (expected {FEATURE_DIM})')
print(f'Feature vector for video {work_list[0]["video_id"]}:')
print(feat)

## 5. Debug pass — process 10 videos

Verify the full loop works end-to-end before committing 20 minutes of compute.

In [ ]:
def extract_all(work_subset, desc='Extracting'):
    X, y, ids, subs = [], [], [], []
    skipped = 0
    for w in tqdm(work_subset, desc=desc):
        feat = video_features(os.path.join(frames_root, w['video_id']))
        if feat is None:
            skipped += 1
            continue
        X.append(feat)
        y.append(w['class_idx'])
        ids.append(w['video_id'])
        subs.append(w['subset'])
    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64), ids, subs, skipped

X, y, ids, subs, skipped = extract_all(work_list[:10], desc='Debug pass')
print(f'\nFeatures shape: {X.shape}')
print(f'Labels shape:   {y.shape}')
print(f'Skipped:        {skipped}')
print(f'First 3 rows of X (rounded):')
print(np.round(X[:3], 3))
print(f'First 3 labels: {y[:3]}')

## 6. Full pass — process all videos and save artifacts

Expect ~15–25 minutes on Colab CPU.

In [ ]:
X, y, ids, subs, skipped = extract_all(work_list, desc='Full extraction')
print(f'\nFinal features shape: {X.shape}')
print(f'Final labels shape:   {y.shape}')
print(f'Skipped:              {skipped}')
print(f'Subset distribution:  {dict(Counter(subs))}')
print(f'Class count:          {len(set(y.tolist()))}')

In [ ]:
# Save artifacts for Member 3
os.makedirs('artifacts', exist_ok=True)

np.save('artifacts/features.npy', X)
np.save('artifacts/labels.npy', y)

splits = {
    'train': [ids[i] for i, s in enumerate(subs) if s == 'train'],
    'val':   [ids[i] for i, s in enumerate(subs) if s == 'val'],
    'test':  [ids[i] for i, s in enumerate(subs) if s == 'test'],
}
with open('artifacts/splits.json', 'w') as f:
    json.dump(splits, f, indent=2)

with open('artifacts/video_ids.json', 'w') as f:
    json.dump(ids, f, indent=2)

with open('artifacts/feature_schema.json', 'w') as f:
    json.dump({
        'feature_dim': FEATURE_DIM,
        'layout': '8 HOOF angle bins + 8 magnitude histogram bins + [mean_mag, std_mag, max_mag, mean_ang]',
        'flow_method': 'Farneback dense optical flow (cv2.calcOpticalFlowFarneback)',
        'flow_resize_wh': list(FLOW_SIZE),
        'angle_bins': ANGLE_BINS,
        'mag_bins': MAG_BINS,
        'aggregation': 'mean across all consecutive frame pairs in the video',
        'num_classes': int(len(set(y.tolist()))),
        'num_videos': int(X.shape[0]),
    }, f, indent=2)

print('Saved artifacts:')
for fn in sorted(os.listdir('artifacts')):
    size = os.path.getsize(f'artifacts/{fn}')
    print(f'  artifacts/{fn}  ({size/1024:.1f} KB)')

In [ ]:
# Bundle and download artifacts + visualizations to commit to the repo
import shutil
from google.colab import files

shutil.make_archive('artifacts_bundle', 'zip', 'artifacts')
shutil.make_archive('visualizations_bundle', 'zip', 'visualizations')

files.download('artifacts_bundle.zip')
files.download('visualizations_bundle.zip')

## 7. Hand-off to Member 3

**What you receive (in `artifacts/`):**

| File | Shape / Type | Purpose |
|---|---|---|
| `features.npy` | `(N, 20) float32` | One row per video — input X for the classifier |
| `labels.npy` | `(N,) int64` | Class index 0–99 — target y |
| `video_ids.json` | list of N strings | Same order as the npy files; lets you trace any row back to its source video |
| `splits.json` | `{train, val, test}` → video_id lists | Use these for train/test split — don't shuffle randomly, the WLASL splits are designed to keep signers separate |
| `feature_schema.json` | metadata | Documents what each feature dimension means |

**Your starter snippet (Step 8):**

```python
import json, numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

X = np.load('artifacts/features.npy')
y = np.load('artifacts/labels.npy')
with open('artifacts/video_ids.json') as f: ids = json.load(f)
with open('artifacts/splits.json') as f: splits = json.load(f)

id_to_row = {v: i for i, v in enumerate(ids)}
tr = [id_to_row[v] for v in splits['train'] if v in id_to_row]
te = [id_to_row[v] for v in splits['test']  if v in id_to_row]

scaler = StandardScaler().fit(X[tr])
model = SVC(kernel='rbf').fit(scaler.transform(X[tr]), y[tr])
print('test acc:', model.score(scaler.transform(X[te]), y[te]))
```

**Important:** Standardize before SVM. The 4 stats columns (indices 16–19) are on a different scale than the histograms (which sum to 1).